In [10]:


# %% [markdown]
# # Phase 1: Data Preparation & Wholesale Query Augmentation
# 
# This notebook:
# 1. Loads the Amazon ESCI dataset from Hugging Face
# 2. Filters to US-English products
# 3. Creates the product corpus with combined text fields
# 4. Augments queries with wholesale/B2B signals
# 5. Creates train/val/test splits
# 6. Saves processed data

# %%

In [11]:
# Imports
import sys
sys.path.append('..')  # Add parent dir to path for src imports

import pandas as pd
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
import random
import json
import os
import yaml

from src.augmentation import WholesaleQueryAugmenter

In [12]:
# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [13]:
# Load config
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully!")
print(f"Project: {config['project']['name']}")


Configuration loaded successfully!
Project: wholesale-product-retrieval


In [14]:
# Create directories
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
print("✓ Data directories created")

✓ Data directories created


In [15]:
# ## 1.1 Load ESCI Dataset

# %%
print("Loading ESCI dataset from Hugging Face...")
print("This may take a few minutes on first run (~2GB download)\n")

dataset = load_dataset("tasksource/esci", trust_remote_code=True)

print("Dataset loaded!")
print("\nAvailable splits:")
for split in dataset:
    print(f"  - {split}: {len(dataset[split]):,} rows")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'tasksource/esci' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading ESCI dataset from Hugging Face...
This may take a few minutes on first run (~2GB download)

Dataset loaded!

Available splits:
  - train: 2,027,874 rows
  - test: 652,490 rows


In [16]:
# Convert to DataFrame
df_examples = dataset['train'].to_pandas()

print(f"\nDataFrame shape: {df_examples.shape}")
print(f"\nColumns:\n{df_examples.columns.tolist()}")


DataFrame shape: (2027874, 14)

Columns:
['example_id', 'query', 'query_id', 'product_id', 'product_locale', 'esci_label', 'small_version', 'large_version', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color', 'product_text']


In [19]:
# Check actual columns in the dataset
print("Columns available:")
print(df_examples.columns.tolist())

print("\nFirst row sample:")
print(df_examples.iloc[0])

Columns available:
['example_id', 'query', 'query_id', 'product_id', 'product_locale', 'esci_label', 'small_version', 'large_version', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color', 'product_text']

First row sample:
example_id                                                              0
query                                                       revent 80 cfm
query_id                                                                0
product_id                                                     B000MOO21W
product_locale                                                         us
esci_label                                                     Irrelevant
small_version                                                           0
large_version                                                           1
product_title           Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...
product_description                                               

In [17]:
# %% [markdown]
# ## 1.2 Explore the Data

# %%
# Label distribution
print("ESCI Label Distribution:")
print("-" * 30)
print(df_examples['esci_label'].value_counts())

print("""
Label meanings:
  E (Exact)      → Exactly what user wants     → relevance: 3
  S (Substitute) → Reasonable alternative      → relevance: 2
  C (Complement) → Complements the query       → relevance: 1
  I (Irrelevant) → Not relevant                → relevance: 0
""")

ESCI Label Distribution:
------------------------------
esci_label
Exact         1342424
Substitute     431593
Irrelevant     197583
Complement      56274
Name: count, dtype: int64

Label meanings:
  E (Exact)      → Exactly what user wants     → relevance: 3
  S (Substitute) → Reasonable alternative      → relevance: 2
  C (Complement) → Complements the query       → relevance: 1
  I (Irrelevant) → Not relevant                → relevance: 0



In [20]:
print("Locale Distribution:")
print("-" * 30)
print(df_examples['product_locale'].value_counts())

Locale Distribution:
------------------------------
product_locale
us    1420372
jp     333112
es     274390
Name: count, dtype: int64


In [21]:
locale = config['data']['esci']['locale']
df_us = df_examples[df_examples['product_locale'] == locale].copy()

In [22]:
print(f"Filtered to locale '{locale}':")
print(f"  Before: {len(df_examples):,} rows")
print(f"  After:  {len(df_us):,} rows")

n_queries = df_us['query'].nunique()
n_products = df_us['product_id'].nunique()
print(f"\nUnique queries:  {n_queries:,}")
print(f"Unique products: {n_products:,}")

Filtered to locale 'us':
  Before: 2,027,874 rows
  After:  1,420,372 rows

Unique queries:  74,888
Unique products: 982,641


In [23]:
# %% [markdown]
# ## 1.4 Create Product Corpus

# %%
# Identify available product columns
product_cols = [
    'product_id', 'product_title', 'product_description',
    'product_bullet_point', 'product_brand', 'product_color', 'product_locale'
]
available_cols = [c for c in product_cols if c in df_us.columns]
print(f"Available product columns: {available_cols}")

# Create deduplicated product dataframe
df_products = df_us[available_cols].drop_duplicates(subset=['product_id']).copy()
print(f"\nProduct corpus size: {len(df_products):,}")

Available product columns: ['product_id', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color', 'product_locale']

Product corpus size: 982,641


In [24]:
# %%
# Create combined searchable text field
def create_product_text(row):
    """Combine product fields into single searchable text."""
    parts = []
    
    if pd.notna(row.get('product_title')):
        parts.append(str(row['product_title']))
    
    if pd.notna(row.get('product_brand')):
        parts.append(f"Brand: {row['product_brand']}")
    
    if pd.notna(row.get('product_color')):
        parts.append(f"Color: {row['product_color']}")
    
    if pd.notna(row.get('product_bullet_point')):
        bullets = str(row['product_bullet_point'])[:500]
        parts.append(bullets)
    
    return " ".join(parts)

print("Creating combined product text...")
df_products['product_text'] = df_products.apply(create_product_text, axis=1)

# Preview
print("\nSample product:")
print("-" * 50)
sample = df_products.iloc[0]
print(f"ID: {sample['product_id']}")
print(f"Title: {sample.get('product_title', 'N/A')}")
print(f"Brand: {sample.get('product_brand', 'N/A')}")
print(f"Combined text preview: {sample['product_text'][:200]}...")


Creating combined product text...

Sample product:
--------------------------------------------------
ID: B000MOO21W
Title: Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceiling Mounted Fan
Brand: Panasonic
Combined text preview: Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceiling Mounted Fan Brand: Panasonic Color: White WhisperCeiling fans feature a totally enclosed condenser motor and a double-tapered, dolphin-shaped bladed b...


In [25]:
# ## 1.5 Create Relevance Labels

# %%
# Map ESCI labels to numeric relevance
# This dataset uses full words: "Exact", "Substitute", "Complement", "Irrelevant"
label_map = {
    'Exact': 3,
    'Substitute': 2,
    'Complement': 1,
    'Irrelevant': 0
}
print(f"Label mapping: {label_map}")

df_labels = df_us[['query', 'product_id', 'esci_label']].copy()
df_labels['relevance'] = df_labels['esci_label'].map(label_map)

# Check for any unmapped labels
unmapped = df_labels[df_labels['relevance'].isna()]['esci_label'].unique()
if len(unmapped) > 0:
    print(f"WARNING: Unmapped labels found: {unmapped}")

print(f"\nLabels DataFrame: {df_labels.shape}")
print("\nRelevance distribution:")
print(df_labels['relevance'].value_counts().sort_index())

Label mapping: {'Exact': 3, 'Substitute': 2, 'Complement': 1, 'Irrelevant': 0}

Labels DataFrame: (1420372, 4)

Relevance distribution:
relevance
0    122273
1     29713
2    280324
3    988062
Name: count, dtype: int64


In [26]:
# %% [markdown]
# ## 1.6 Wholesale Query Augmentation

# %%
# Initialize augmenter
augmenter = WholesaleQueryAugmenter(config_path='../configs/config.yaml')

# Get unique queries
unique_queries = df_labels['query'].unique().tolist()
print(f"Original unique queries: {len(unique_queries):,}")

Original unique queries: 74,888


In [27]:
# %%
# Augment queries
aug_config = config['augmentation']
augmented_map = augmenter.augment_batch(
    queries=unique_queries,
    n_augmentations=aug_config['augmentations_per_query'],
    augment_ratio=aug_config['ratio']
)

print(f"\nAugmented {len(augmented_map):,} queries")
print(f"Total new queries: {sum(len(v) for v in augmented_map.values()):,}")



Augmented 18,722 queries
Total new queries: 43,118


In [28]:
# Show examples
print("\nAugmentation Examples:")
print("-" * 50)
for orig, augs in list(augmented_map.items())[:5]:
    print(f"\nOriginal: '{orig}'")
    for aug in augs:
        print(f"  → '{aug}'")



Augmentation Examples:
--------------------------------------------------

Original: 'dental care'
  → 'dental care bulk pack'
  → 'dental care for resale'

Original: 'travel keurig coffee maker mini with case'
  → 'bulk travel keurig coffee maker mini with case'
  → 'travel keurig coffee maker mini with case multi pack'
  → 'travel keurig coffee maker mini with case 50 count'

Original: 'advocare 24 day challenge bundle'
  → 'advocare 24 day challenge bundle bulk'
  → 'advocare 24 day challenge bundle multi pack'

Original: 'pullover ski coat for women'
  → 'bulk pullover ski coat for women'
  → 'pullover ski coat for women retail pack'

Original: 'axis camera'
  → 'axis camera retail pack'
  → 'axis camera wholesale'
  → 'axis camera pack of 24'


In [31]:
# %% [markdown]
# ## 1.7 Create Augmented Labels DataFrame

# %%
# Build augmented labels (inherit from original)
augmented_rows = []

for orig_query, aug_queries in tqdm(augmented_map.items(), desc="Building augmented labels"):
    orig_labels = df_labels[df_labels['query'] == orig_query]
    
    for aug_query in aug_queries:
        for _, row in orig_labels.iterrows():
            augmented_rows.append({
                'query': aug_query,
                'original_query': orig_query,
                'product_id': row['product_id'],
                'esci_label': row['esci_label'],
                'relevance': row['relevance'],
                'is_augmented': True
            })

df_augmented = pd.DataFrame(augmented_rows)

# Add flags to original
df_labels['original_query'] = df_labels['query']
df_labels['is_augmented'] = False

# Combine original + augmented  ← ADD THIS
df_labels_combined = pd.concat([df_labels, df_augmented], ignore_index=True)

print(f"\nCombined Labels:")
print(f"  Original rows:  {len(df_labels):,}")
print(f"  Augmented rows: {len(df_augmented):,}")
print(f"  Total rows:     {len(df_labels_combined):,}")

Building augmented labels: 100%|██████████| 18722/18722 [10:57<00:00, 28.49it/s]



Combined Labels:
  Original rows:  1,420,372
  Augmented rows: 819,749
  Total rows:     2,240,121


In [32]:
# %% [markdown]
# ## 1.8 Train/Val/Test Split

# %%
# Split by unique queries (prevents data leakage)
all_queries = df_labels_combined['query'].unique().tolist()
random.shuffle(all_queries)

split_ratios = config['data']['splits']
n_total = len(all_queries)
n_train = int(n_total * split_ratios['train'])
n_val = int(n_total * split_ratios['val'])

train_queries = set(all_queries[:n_train])
val_queries = set(all_queries[n_train:n_train + n_val])
test_queries = set(all_queries[n_train + n_val:])

print(f"Split sizes:")
print(f"  Train: {len(train_queries):,} queries")
print(f"  Val:   {len(val_queries):,} queries")
print(f"  Test:  {len(test_queries):,} queries")

Split sizes:
  Train: 82,602 queries
  Val:   17,700 queries
  Test:  17,701 queries


In [33]:
# %%
# Assign splits
def assign_split(query):
    if query in train_queries:
        return 'train'
    elif query in val_queries:
        return 'val'
    return 'test'

df_labels_combined['split'] = df_labels_combined['query'].apply(assign_split)

print("\nRows per split:")
print(df_labels_combined['split'].value_counts())



Rows per split:
split
train    1570629
val       335030
test      334462
Name: count, dtype: int64


In [34]:
# %% [markdown]
# ## 1.9 Save Processed Data

# %%
output_dir = '../data/processed'

# Save products
products_path = f'{output_dir}/products.parquet'
df_products.to_parquet(products_path, index=False)
print(f"✓ Saved {products_path} ({len(df_products):,} products)")

# Save all labels
labels_path = f'{output_dir}/labels.parquet'
df_labels_combined.to_parquet(labels_path, index=False)
print(f"✓ Saved {labels_path} ({len(df_labels_combined):,} rows)")

# Save split-specific files
for split in ['train', 'val', 'test']:
    split_df = df_labels_combined[df_labels_combined['split'] == split]
    split_path = f'{output_dir}/labels_{split}.parquet'
    split_df.to_parquet(split_path, index=False)
    print(f"✓ Saved {split_path} ({len(split_df):,} rows)")

# Save query splits as JSON
query_splits = {
    'train': list(train_queries),
    'val': list(val_queries),
    'test': list(test_queries)
}
with open(f'{output_dir}/query_splits.json', 'w') as f:
    json.dump(query_splits, f)
print(f"✓ Saved query_splits.json")


✓ Saved ../data/processed/products.parquet (982,641 products)
✓ Saved ../data/processed/labels.parquet (2,240,121 rows)
✓ Saved ../data/processed/labels_train.parquet (1,570,629 rows)
✓ Saved ../data/processed/labels_val.parquet (335,030 rows)
✓ Saved ../data/processed/labels_test.parquet (334,462 rows)
✓ Saved query_splits.json


In [35]:
# %% [markdown]
# ## 1.10 Summary

# %%
print("\n" + "=" * 60)
print("PHASE 1 COMPLETE")
print("=" * 60)

print(f"""
Dataset Summary
───────────────
Products:           {len(df_products):,}
Total queries:      {len(all_queries):,}
  - Original:       {len(unique_queries):,}
  - Augmented:      {len(all_queries) - len(unique_queries):,}

Label Distribution:
  - Exact (3):      {(df_labels_combined['relevance'] == 3).sum():,}
  - Substitute (2): {(df_labels_combined['relevance'] == 2).sum():,}
  - Complement (1): {(df_labels_combined['relevance'] == 1).sum():,}
  - Irrelevant (0): {(df_labels_combined['relevance'] == 0).sum():,}

Splits:
  - Train: {len(train_queries):,} queries ({split_ratios['train']*100:.0f}%)
  - Val:   {len(val_queries):,} queries ({split_ratios['val']*100:.0f}%)
  - Test:  {len(test_queries):,} queries ({split_ratios['test']*100:.0f}%)

Output Files (data/processed/):
  ✓ products.parquet
  ✓ labels.parquet
  ✓ labels_train.parquet
  ✓ labels_val.parquet
  ✓ labels_test.parquet
  ✓ query_splits.json

Next: Run 02_query_understanding.ipynb
""")


PHASE 1 COMPLETE

Dataset Summary
───────────────
Products:           982,641
Total queries:      118,003
  - Original:       74,888
  - Augmented:      43,115

Label Distribution:
  - Exact (3):      1,558,094
  - Substitute (2): 442,427
  - Complement (1): 47,340
  - Irrelevant (0): 192,260

Splits:
  - Train: 82,602 queries (70%)
  - Val:   17,700 queries (15%)
  - Test:  17,701 queries (15%)

Output Files (data/processed/):
  ✓ products.parquet
  ✓ labels.parquet
  ✓ labels_train.parquet
  ✓ labels_val.parquet
  ✓ labels_test.parquet
  ✓ query_splits.json

Next: Run 02_query_understanding.ipynb



In [36]:
import pandas as pd

# Check products
products = pd.read_parquet('../data/processed/products.parquet')
print(f"Products loaded: {len(products):,}")
print(products.head(2))

# Check labels
labels = pd.read_parquet('../data/processed/labels_train.parquet')
print(f"\nTrain labels loaded: {len(labels):,}")
print(labels.head(2))

Products loaded: 982,641
   product_id                                      product_title  \
0  B000MOO21W  Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...   
1  B07X3Y6B1V  Homewerks 7141-80 Bathroom Fan Integrated LED ...   

  product_description                               product_bullet_point  \
0                None  WhisperCeiling fans feature a totally enclosed...   
1                None  OUTSTANDING PERFORMANCE: This Homewerk's bath ...   

  product_brand product_color product_locale  \
0     Panasonic         White             us   
1     Homewerks        80 CFM             us   

                                        product_text  
0  Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...  
1  Homewerks 7141-80 Bathroom Fan Integrated LED ...  

Train labels loaded: 1,570,629
                              query  product_id  esci_label  relevance  \
0        bathroom fan without light  B000MOO21W       Exact          3   
1  12 inch bathroomwall mounted fan  B076Q7V5WX  Ir